# Projeto: Visualização da Informação

## Objetivo do Projeto

O objetivo deste projeto é analisar o catálogo da Netflix para entender como a plataforma evoluiu ao longo dos anos e como os seus filmes e séries estão distribuídos pelo mundo.

### O que foi feito:
* **Limpeza dos Dados:** Organização de colunas complexas
* **Análise de Tendências:** Descoberta de qual tipo me mídia está em alta
* **Mapeamento Global:** Criação de um mapa múndi interativo para mostrar visualmente quais países produzem mais conteúdo.

### Fonte

https://www.kaggle.com/datasets/shivamb/netflix-shows


## Importações

In [ ]:
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

## Funções Auxiliares

In [ ]:
# Função para Visualização, retorna um DataFrame
def EDA(data):
    eda_df = pd.DataFrame({
        'coluna': df.columns,
        'dtype': df.dtypes.values,
        'nulos': df.isnull().sum().values,
        'nulos_%': (df.isnull().sum() / len(df) * 100).round(2).values,
        'unicos': df.nunique().values
    })
    return eda_df

# Função para quebrar o formato de Data (Experiências anteriores com Gradient Boosting e Power BI demonstraram grande potencial nesse método)
def quebra_date(df):
    df["date_added"] = df["date_added"].str.strip()

    data = pd.to_datetime(
        df["date_added"],
        format="%B %d, %Y",
        errors="coerce"
    )

    df["day_added"] = data.dt.day
    df["month_added"] = data.dt.month
    df["year_added"] = data.dt.year
    return df

# Remapeamento de valores utilizando um dicionário.
def remapper(df):
    meses = {
    1:"Jan", 2:"Fev", 3:"Mar", 4:"Abr",
    5:"Mai", 6:"Jun", 7:"Jul", 8:"Ago",
    9:"Set", 10:"Out", 11:"Nov", 12:"Dez"
    }

    df["month_added"] = df["month_added"].map(meses)
    return df

def dropper_cols(df):
    # Remoção de linhas com poucos nulos
    # Linhas com muitos nulos foram deixadas, pois sua remoção pode alterar o resultado
    df.dropna(subset=["date_added"], inplace=True)
    df.dropna(subset=["rating"], inplace=True)
    df.dropna(subset=["duration"], inplace=True)
    return df



## Análise Exploratória dos Dados (EDA) + Pré Processamento de Dados

In [ ]:

# Carregamento dos dados
df = pd.read_csv("/content/netflix_titles.csv")

# Visualição Pré Processamento (Tire do comentário para ver)
'''
eda = EDA(df)
print("Pequena amostra do Dataset")
display(df.head())
print("Informações do Dataset")
display(eda)
'''

# Tratamento de Dados
df = quebra_date(df)
df = dropper_cols(df)
df = remapper(df)

# Visualização Pós Processamento
eda = EDA(df)
print("Pequena amostra do Dataset após o tratamento de dados")
display(df.head())
print("Informações do Dataset")
display(eda)

FileNotFoundError: [Errno 2] No such file or directory: '/content/netflix_titles.csv'

## Gráfico de Barras

In [ ]:
############################
# Com Matplotlib
############################

# Contagem de categorias
contagem = df["type"].value_counts()

# Gráfico
plt.figure(figsize=(8, 5))
plt.bar(contagem.index, contagem.values)

# Títulos
plt.title("Distribuição Filme / TV Show")
plt.xlabel("Tipo")
plt.ylabel("Quantidade")

plt.show()

In [ ]:
################
# Com Seaborn
################
plt.figure(figsize=(8, 5))

ordem_meses = [
    "Jan", "Fev", "Mar", "Abr",
    "Mai", "Jun", "Jul", "Ago",
    "Set", "Out", "Nov", "Dez"
]

sns.countplot(
    data=df,
    x="month_added",
    order = ordem_meses
)

plt.title("Distribuição por Mês")
plt.xlabel("Mês")
plt.ylabel("Quantidade")

plt.show()

NameError: name 'df' is not defined

<Figure size 800x500 with 0 Axes>

## Visualização Temporal

In [ ]:
# Gráfico Temporal, mostrando a diferença entre adição de FILMES / TV SHOWS durante os anos

dados = (
    df.groupby(["year_added", "type"])
      .size()
      .reset_index(name="quantidade")
)

plt.figure(figsize=(10, 5))

sns.lineplot(
    data=dados,
    x="year_added",
    y="quantidade",
    hue="type",
    marker="o"
)

plt.legend(title="Mídia", frameon=True)
plt.title("Títulos adicionados por ano")
plt.xlabel("Ano")
plt.ylabel("Quantidade")

plt.show()

## Visualização Geográfica

In [ ]:
limites = [0, 5, 20, 50, 150, 500, 1500, 2500, 4000]

nomes_grupos = [
    "Mínima (1-5)",
    "Muito Baixa (6-20)",
    "Baixa (21-50)",
    "Baixa-Média (51-150)",
    "Média (151-500)",
    "Média-Alta (501-1500)",
    "Alta (1501-2500)",
    "Extrema (>2500)"
]


df_choropleth = df['country'].value_counts().reset_index()
df_choropleth.columns = ['country', 'quantidade']


df_choropleth["Faixa"] = pd.cut(
    df_choropleth["quantidade"],
    bins=limites,
    labels=nomes_grupos
)

fig = px.choropleth(
    df_choropleth.sort_values("quantidade"),
    locations="country",
    locationmode="country names",
    color="Faixa",
    hover_name="country",
    hover_data={"quantidade": True},
    color_discrete_sequence=px.colors.sequential.Reds[1:],
    title="Distribuição de Títulos por Países"
)

fig.update_layout(legend_title_text="Faixa de Títulos")
fig.show()

## Conclusão

- Filmes representam a maior parte dos títulos adicionados ao catálogo da Netflix.
- Até 2016, o ritmo de adição de filmes e séries manteve-se relativamente equilibrado.
- Após 2016, houve um aumento expressivo na incorporação de filmes em comparação às séries.
- Os Estados Unidos apresentam participação significativamente superior aos demais países, caracterizando um outlier na distribuição geográfica do conteúdo.
- Índia e Reino Unido aparecem como os principais mercados secundários, ocupando a segunda e a terceira colocação em número de títulos.